# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Paper:** *Which pages lose the most clicks for their Google position? A position-adjusted review queue, tested on websites it never saw* — Janudi Rupasinghe.

This notebook is the short companion to the deployed paper. It does **not** re-train anything: the heavy work lives in `w01`–`w07`. Here each section restates the paper in a few lines and the code cell below it loads the committed receipts (`work/outputs/playbook_metrics.json`, `work/figures/*.png`) or shows the numbers copied from the notebook that computed them, with that notebook named. No data token is needed to run it.

In [ ]:
# Setup: load the committed receipts straight from the public repo (no data token needed).
import json, urllib.request
import pandas as pd
from IPython.display import Image, display

RAW = "https://raw.githubusercontent.com/jjcodes999/flyrank-ml-repo/main/"

def fetch(path):
    with urllib.request.urlopen(RAW + path) as r:
        return r.read()

metrics = json.loads(fetch("work/outputs/playbook_metrics.json"))
print("Loaded playbook_metrics.json with keys:", ", ".join(metrics.keys()))

## 1. Question

*The research question and the decision it supports.*

**Question:** which pages lose the most clicks for their Google position, and which should an editor review first?

**Decision it supports:** the order in which a person reviews pages for a click-through problem (title and description shown in Google). The output is a ranked review queue, not an automatic fix.

**Prediction target (proxy):** a page is `still_under = 1` if its CTR in 16–31 March 2026 is still below the median CTR of its position band. Inputs come only from 1–15 March.

In [ ]:
# The scale of the question, from the committed receipts.
print("Data:", metrics["data"])
print("Pages scored:", f'{metrics["pages_scored"]:,}', "| clients:", metrics["clients"])
print("Base rate (share still under-performing later):", metrics["base_rate"])
print("Pages in the review queue:", f'{metrics["pages_in_queue"]:,}')

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

- **Release:** pseudonymized FlyRank internship warehouse, build `flyrank_pseudonymized_warehouse_release_v20260703` (Hugging Face, gated).
- **Table:** `fact_content_daily_performance`, partition `month=2026-03` only (one mid-panel month). `dim_content` used once, for page age.
- **Windows:** features 1–15 March; outcome 16–31 March.
- **Kept:** rows with `gsc_data_available IS TRUE`; pages with ≥100 impressions in the first half and ≥50 in the second.
- **Excluded:** any second-half number as input (future); GA4 columns (happen after a click); `gsc_sum_position` (undocumented); June 2026 (sealed final month); identifiers as inputs (used only to group by client). No client names, URLs or queries are used or shown.

In [ ]:
# From the March partition to the analysis frame (numbers computed in w03_data_contract and w06_validation_audit).
funnel = pd.DataFrame([
    ["March 2026 daily rows", 9_841_378, "w03_data_contract"],
    ["Rows with search data (gsc_data_available TRUE)", 3_611_061, "w03_data_contract"],
    ["Pages with >=100 impressions in 1-15 March", 77_540, "w06_validation_audit"],
    ["Pages also with >=50 impressions in 16-31 March (final frame)", metrics["pages_scored"], "playbook_metrics.json"],
], columns=["step", "count", "source"])
display(funnel)
print(f"Share of rows with search data: {3_611_061 / 9_841_378:.1%}")
print(f"Pages removed by the second-half filter: {77_540 - metrics['pages_scored']:,} "
      f"({(77_540 - metrics['pages_scored']) / 77_540:.1%})")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

- **Unit:** one page. Decision moment: the morning of 16 March.
- **Features (1–15 March):** impressions, CTR, impression-weighted average position, days visible, and the **click gap** = impressions × (median CTR of the page's position band − page CTR) ÷ 100.
- **Label:** CTR in 16–31 March still below its band's median (base rate ≈ 0.35).
- **Baseline rule (frozen before modelling):** top-20 position, ≥1,000 impressions, below-typical CTR; ranked by click gap.
- **Models:** logistic regression (scaled, log-squashed skewed features); random forest (200 trees, min 20 pages per leaf, seed 42).
- **Validation:** `GroupKFold(5)` grouped by client, so every score comes from websites the model never saw. Metric: Precision@K (10/20/50/100), AUC as a whole-list check.
- **Leakage checks:** a planted leak column takes AUC to 1.000 (the harness works); dropping position or click gap causes no collapse; the 2.2% population filter and the rule's same-month cutoff are disclosed as optimistic.

In [ ]:
# The leakage audit (computed in w06_validation_audit): random forest, client-grouped folds.
leak = pd.DataFrame([
    ["Final model", 0.805, 0.98, 0.884],
    ["+ deliberately leaked column", 1.000, 1.00, 1.000],
    ["- position_h1", 0.799, 0.98, 0.866],
    ["- missed_clicks_h1", 0.806, 0.94, 0.868],
], columns=["variant", "AUC", "P@20", "P@100"])
display(leak)

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

On unseen client websites, the rule's top 20 were still under-performing later 93% of the time (base rate 35%). The random forest was slightly higher at the very top (98%) but not deeper, and its top 20 pointed to about half the click gap (362 vs 755 estimated missed clicks). Logistic regression did not beat the rule at any K. A random split would have made logistic regression look better than the rule at P@50/P@100; the client-grouped split removes that.

In [ ]:
# Precision@K, mean over 5 client-grouped folds (rule and forest from the receipts; LR and AUC from w05_model).
pk = pd.DataFrame(metrics["precision_at_k_grouped_5fold"]).T
pk.loc["Logistic regression"] = [0.94, 0.92, 0.896, 0.876]
pk["AUC"] = pd.Series({"Random pick (base rate)": 0.50, "Rule (the queue)": 0.566,
                       "Random forest (confidence column)": 0.805, "Logistic regression": 0.809})
pk = pk.loc[["Random pick (base rate)", "Rule (the queue)", "Logistic regression", "Random forest (confidence column)"]]
display(pk.round(3))

# Value of each method's top 20 per fold (w05_model)
value = pd.DataFrame({"median impressions": [21_342, 14_397, 9_117], "click gap covered": [755, 671, 362]},
                     index=["Rule", "Logistic regression", "Random forest"])
display(value)

## 5. Limitations

*What this work cannot claim.*

- One month only (March 2026); stability across months is untested.
- The label is a proxy: it shows a low CTR persisted, not that editing a page would raise clicks (that needs an experiment).
- Narrow population: 75,817 pages from 37 clients, about 23% of March pages; absence from the queue is not a health check.
- Concentration: two clients hold 72% of the queue and one fold is a single client.
- The rule's cutoff was chosen on the same month, and the population filter uses the outcome window (2.2% of pages): both slightly optimistic.
- Position is an average over many searches; no confidence intervals, only fold-to-fold variation.

In [ ]:
# How uneven the client-grouped folds are (w06_validation_audit): why methods are compared inside each fold.
folds = pd.DataFrame({"fold": [1, 2, 3, 4, 5],
                      "test clients": [1, 3, 11, 11, 11],
                      "base rate": [0.44, 0.50, 0.26, 0.32, 0.24]})
display(folds)
print("Random forest P@20 vs rule, fold by fold: 3 wins, 1 tie, 1 loss")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. Clear the checks first: `check_data_quality` (average position below 1) and `check_tracking_first` (0 clicks), then review 20–50 pages per cycle from the top with `review_title_and_meta`.
2. Review dominant clients at site level first (templates, brand searches, search-result layout).
3. Look at the real search result before editing anything.
4. Leave pages outside the queue alone (low volume: monitor; beyond position 20: a ranking problem, not a snippet problem).
5. Rebuild monthly and re-check the rule if the base rate moves by more than 0.10, P@20 falls below 0.80, zero-click share rises above 0.25, or one client exceeds 60% of the top 50.

**Never automate:** title/description rewrites or publishing; deleting, merging or redirecting; acting on data or tracking cases before the check; use outside the population; judging people; reporting the click gap as clicks that will be won.

In [ ]:
# The playbook from the receipts.
display(pd.Series(metrics["queue_by_action"], name="pages").to_frame())
display(pd.DataFrame(metrics["archetypes"]).T)
cov = pd.DataFrame(metrics["missed_clicks_covered"]).T
cov.index.name = "top N pages"
display(cov)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper embeds the two committed figures below (made in `w07_action_playbook`) and the tables above. The paper itself is `docs/index.html` in this repo, served by GitHub Pages.

In [ ]:
# Figure 1 and Figure 2 of the paper, loaded from work/figures/.
for name in ["precision_at_k.png", "click_gap_coverage.png"]:
    print(name)
    display(Image(data=fetch("work/figures/" + name), width=700))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
